# 02807 Computational Tools for Data Science: AVA Data Cleaning & Feature Extraction

This notebook implements the data ingestion, cleaning, structural feature calculation, and CLIP embedding extraction pipeline for the **AVA Aesthetic Dataset** (`trojblue/AVA-Huggingface`).

### Why We Extract Both Feature Types:
1. **CLIP Embeddings (512-dim):** Captures high-level semantic meaning (*what* is in the photo: subject matter, scene type, mood).
2. **Structural Features (12-dim):** Captures low-level photographic execution (*how* the photo is shot: luminance contrast, brightness, center-to-edge sharpness ratio / bokeh proxy, RGB color distributions, and aspect ratio). These low-dimensional features are essential for our **Course Topic 1 (General Clustering & Davies-Bouldin Index)** and bridge AVA's visual patterns with Unsplash's EXIF camera metadata.

## Step 1: Install & Import Dependencies
In Colab, select a GPU runtime, then run the installation cell. The dataset and CLIP model are public; Hugging Face login is not required. The next cell mounts Google Drive for persistent checkpoints. Locally, it writes to a separate `data/ava_full_features` folder and does not overwrite the existing sample.

In [ ]:
%pip install datasets transformers torch pillow numpy pyarrow huggingface_hub tqdm

In [ ]:
import importlib.util
from pathlib import Path

IN_COLAB = (importlib.util.find_spec('google') is not None
            and importlib.util.find_spec('google.colab') is not None)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/ava_full_features')
else:
    OUTPUT_DIR = Path('data/ava_full_features')

DATASET_NAME = 'trojblue/AVA-Huggingface'
MODEL_ID = 'openai/clip-vit-base-patch32'
BATCH_SIZE = 64
CHECKPOINT_EVERY = 1024  # Unique images processed, including rejected images.
MAX_SAMPLES = None  # Optional limit on accepted images, including saved ones.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Persistent output folder: {OUTPUT_DIR}')

In [ ]:
import io
import json
import os
from collections import Counter

import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import requests
import torch
from PIL import Image, ImageChops, UnidentifiedImageError
from datasets import load_dataset, Dataset, Image as DatasetImage
from huggingface_hub import HfApi
from transformers import CLIPProcessor, CLIPModel
from tqdm.auto import tqdm

## Step 2: Configure Hardware & Load Frozen CLIP Backbone
Select a Colab GPU for this step. Hardware selection also supports Apple Silicon (`mps`) or CPU. Dataset/model revisions are pinned in `run_config.json` so resumed runs use the same inputs. Changing the dataset or feature definition requires a new output folder.

In [ ]:
config_path = OUTPUT_DIR / 'run_config.json'
expected_config = {'dataset': DATASET_NAME, 'model': MODEL_ID, 'features_version': 1}
if config_path.exists():
    run_config = json.loads(config_path.read_text())
    if any(run_config.get(key) != value for key, value in expected_config.items()):
        raise ValueError('Output folder belongs to a different extraction run. Use a new OUTPUT_DIR.')
else:
    if list(OUTPUT_DIR.glob('part-*.parquet')):
        raise ValueError('Checkpoint files exist without run_config.json; restore the config first.')
    api = HfApi()
    run_config = {**expected_config,
                  'dataset_revision': api.dataset_info(DATASET_NAME).sha,
                  'model_revision': api.model_info(MODEL_ID).sha}
    temporary_config = config_path.with_suffix('.json.tmp')
    temporary_config.write_text(json.dumps(run_config, indent=2))
    os.replace(temporary_config, config_path)

device = (
    "mps" if torch.backends.mps.is_available() 
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
print(f"Using device: {device}")

model = CLIPModel.from_pretrained(
    MODEL_ID, revision=run_config['model_revision']
).to(device).eval()
processor = CLIPProcessor.from_pretrained(MODEL_ID, revision=run_config['model_revision'])

## Step 3: Define Image Cleaning & Structural Feature Functions
- **`trim_uniform_borders`**: Many AVA images come from DPChallenge photography contests and have artificial solid black or white borders added by users. We strip these borders so they don't corrupt color histograms or contrast calculations.
- **`extract_structural_features`**: Computes 9 core pixel-level metrics (brightness, RMS contrast, center-vs-edge energy ratio for subject isolation/bokeh, and RGB means/standard deviations), which we combine with aspect ratio, width, and height.

In [ ]:
def trim_uniform_borders(img: Image.Image, tolerance: int = 15) -> Image.Image:
    """Trims artificial solid black/white contest borders from AVA images."""
    bg = Image.new(img.mode, img.size, img.getpixel((0, 0)))
    diff = ImageChops.difference(img, bg)
    diff = ImageChops.add(diff, diff, 2.0, -tolerance)
    bbox = diff.getbbox()
    return img.crop(bbox) if bbox else img


def extract_structural_features(img: Image.Image) -> list[float]:
    """Extracts lightweight color and structural descriptors for clustering & PCA."""
    arr = np.asarray(img.resize((128, 128)), dtype=np.float32) / 255.0
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]

    # Luminance (perceived brightness) and RMS contrast
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    contrast = float(np.std(lum))
    brightness = float(np.mean(lum))

    # Center vs. global energy ratio (proxy for subject isolation / shallow depth of field)
    center_lum = float(np.std(lum[32:96, 32:96]))
    center_ratio = center_lum / (contrast + 1e-5)

    return [
        brightness,
        contrast,
        center_ratio,
        float(np.mean(r)), float(np.mean(g)), float(np.mean(b)),
        float(np.std(r)), float(np.std(g)), float(np.std(b)),
    ]

## Step 4: Stream, Clean, and Extract Features in Batches
Stream **every source split**, deduplicating image IDs across splits. Images are decoded only after checking saved IDs, so restarting does not repeat CLIP computation. Restarting still rescans the source stream and may transfer earlier Parquet/image bytes; it is not a byte-offset download resume.

Every `CHECKPOINT_EVERY` unique images, save an atomic `part-*.parquet` checkpoint containing accepted features and rejection statuses. Only one checkpoint's records and one CLIP batch are held in memory. Rerun the notebook with the same output folder after a disconnect: saved parts are validated and skipped. Up to one unfinished checkpoint may need reprocessing. Do not run two extraction sessions against the same folder concurrently.

**Cleaning rules applied:**
1. Force 3-channel RGB (`img.convert("RGB")`) to handle grayscale and 4-channel RGBA images.
2. Trim uniform contest borders.
3. Filter out corrupted images, tiny thumbnails ($< 128 \times 128$), and extreme banner aspect ratios ($> 4:1$ or $< 1:4$).
4. Verify at least 50 total votes and compute both the weighted **mean aesthetic score ($\mu$)** and **vote standard deviation ($\sigma$)**.

In [ ]:
FEATURE_SCHEMA = pa.schema([
    ('image_id', pa.string()), ('mean_score', pa.float64()),
    ('std_score', pa.float64()), ('total_votes', pa.int64()),
    ('rating_counts', pa.list_(pa.int64())),
    ('structural_features', pa.list_(pa.float32())),
    ('clip_embedding', pa.list_(pa.float32())),
])
CHECKPOINT_SCHEMA = FEATURE_SCHEMA.append(pa.field('status', pa.string()))
# Accept checkpoints written before nullable vectors switched to regular lists.
LEGACY_CHECKPOINT_SCHEMA = CHECKPOINT_SCHEMA.set(
    CHECKPOINT_SCHEMA.get_field_index('structural_features'),
    pa.field('structural_features', pa.list_(pa.float32(), 12)),
).set(
    CHECKPOINT_SCHEMA.get_field_index('clip_embedding'),
    pa.field('clip_embedding', pa.list_(pa.float32(), 512)),
)


def open_stream_image(payload):
    if payload is None:
        return None
    if payload.get('bytes') is not None:
        source = io.BytesIO(payload['bytes'])
    else:
        source = payload.get('path')
        if source is None:
            return None
        if source.startswith(('https://', 'http://')):
            response = requests.get(source, timeout=60)
            response.raise_for_status()
            source = io.BytesIO(response.content)
    with Image.open(source) as image:
        return image.convert('RGB')


def extract_with_checkpoints(streams):
    if BATCH_SIZE < 1 or CHECKPOINT_EVERY < 1 or (MAX_SAMPLES is not None and MAX_SAMPLES < 1):
        raise ValueError('Batch/checkpoint sizes and an optional sample limit must be positive.')
    seen_ids, status_counts = set(), Counter()
    parts = sorted(OUTPUT_DIR.glob('part-*.parquet'))
    for part_index, part in enumerate(parts):
        if part.name != f'part-{part_index:06d}.parquet':
            raise ValueError('Checkpoint numbering has gaps; restore missing files before resuming.')
        stored_schema = pq.read_schema(part)
        if not any(stored_schema.equals(schema) for schema in (CHECKPOINT_SCHEMA, LEGACY_CHECKPOINT_SCHEMA)):
            raise ValueError(f'Unexpected checkpoint schema: {part}')
        saved = pq.read_table(part, columns=['image_id', 'status']).to_pydict()
        for image_id, status in zip(saved['image_id'], saved['status']):
            if image_id is None or image_id in seen_ids:
                raise ValueError(f'Duplicate or missing ID in saved checkpoints: {image_id}')
            if status not in {'accepted', 'missing_image', 'corrupt_image', 'geometry', 'votes'}:
                raise ValueError(f'Unknown saved status: {status}')
            seen_ids.add(image_id)
            status_counts[status] += 1
    print(f'Resuming {len(parts)} checkpoints: {status_counts["accepted"]:,} accepted images saved.')
    records, batch_imgs, batch_meta = [], [], []
    duplicate_count = 0

    def embed_batch(pbar):
        if not batch_imgs:
            return
        with torch.no_grad():
            inputs = processor(images=batch_imgs, return_tensors='pt').to(device)
            vision = model.vision_model(pixel_values=inputs['pixel_values'])
            embeddings = model.visual_projection(vision.pooler_output)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=-1)
            embeddings = embeddings.cpu().numpy().astype(np.float32)
        if embeddings.shape != (len(batch_meta), 512) or not np.isfinite(embeddings).all():
            raise ValueError('CLIP returned invalid embeddings.')
        for meta, embedding in zip(batch_meta, embeddings):
            records.append({**meta, 'clip_embedding': embedding.tolist(), 'status': 'accepted'})
        status_counts['accepted'] += len(batch_meta)
        pbar.update(len(batch_meta))
        batch_imgs.clear()
        batch_meta.clear()

    def save_checkpoint(pbar):
        embed_batch(pbar)
        if not records:
            return
        part = OUTPUT_DIR / f'part-{len(parts):06d}.parquet'
        temporary = part.with_suffix('.parquet.tmp')
        pq.write_table(pa.Table.from_pylist(records, schema=CHECKPOINT_SCHEMA), temporary,
                       compression='zstd')
        os.replace(temporary, part)
        parts.append(part)
        records.clear()
        tqdm.write(f'Saved {part.name}: {status_counts["accepted"]:,} accepted images total.')

    with tqdm(total=MAX_SAMPLES, initial=status_counts['accepted'], desc='Cleaning & Embedding AVA') as pbar:
        stop = False
        for split, stream in streams.items():
            print(f'Streaming split: {split}')
            for row in stream.cast_column('image', DatasetImage(decode=False)):
                if MAX_SAMPLES is not None and status_counts['accepted'] + len(batch_meta) >= MAX_SAMPLES:
                    stop = True
                    break
                if row['image_id'] is None:
                    raise ValueError('Source row has no image ID.')
                image_id = str(row['image_id'])
                if image_id in seen_ids:
                    duplicate_count += 1
                    continue
                counts = np.asarray(row['rating_counts'], dtype=np.float64)
                if (counts.shape != (10,) or not np.isfinite(counts).all()
                        or (counts < 0).any() or not np.equal(counts, np.floor(counts)).all()):
                    raise ValueError(f'Invalid rating counts for image {image_id}')
                total_votes = int(counts.sum())
                status = None
                if total_votes < 50:
                    status = 'votes'
                else:
                    try:
                        img = open_stream_image(row['image'])
                    except requests.RequestException:
                        raise
                    except (UnidentifiedImageError, OSError) as exc:
                        tqdm.write(f'Corrupt image {image_id}: {exc}')
                        status = 'corrupt_image'
                    if status is None:
                        if img is None:
                            status = 'missing_image'
                        else:
                            img = trim_uniform_borders(img)
                            w, h = img.size
                            aspect_ratio = w / h
                            if w < 128 or h < 128 or not 0.25 <= aspect_ratio <= 4.0:
                                status = 'geometry'
                seen_ids.add(image_id)
                if status is not None:
                    tqdm.write(f'Rejected image {image_id}: {status}')
                    records.append({'image_id': image_id, 'status': status})
                    status_counts[status] += 1
                else:
                    scores = np.arange(1, 11, dtype=np.float64)
                    probabilities = counts / total_votes
                    mean_score = float(probabilities @ scores)
                    std_score = float(np.sqrt(np.sum(probabilities * (scores - mean_score) ** 2)))
                    batch_imgs.append(img)
                    batch_meta.append({
                        'image_id': image_id, 'mean_score': mean_score, 'std_score': std_score,
                        'total_votes': total_votes, 'rating_counts': counts.astype(np.int64).tolist(),
                        'structural_features': extract_structural_features(img) + [aspect_ratio, float(w), float(h)],
                    })
                    if len(batch_imgs) >= BATCH_SIZE:
                        embed_batch(pbar)
                if len(records) + len(batch_meta) >= CHECKPOINT_EVERY:
                    save_checkpoint(pbar)
            if stop:
                break
        save_checkpoint(pbar)
    print(f'Finished: {dict(status_counts)}; skipped {duplicate_count:,} saved/repeated IDs.')
    return status_counts


streams = load_dataset(DATASET_NAME, revision=run_config['dataset_revision'], streaming=True)
status_counts = extract_with_checkpoints(streams)

## Step 5: Export Accepted Features to One Parquet File
Combine checkpoints one at a time, excluding rejected images. The result has the same seven columns as the existing modeling dataset. It stays in the persistent output folder. Run this cell after extraction finishes; checkpoints remain available for resume. A temporary file is renamed only after export succeeds.

In [ ]:
parquet_path = OUTPUT_DIR / 'ava_cleaned_features.parquet'
temporary_export = parquet_path.with_suffix('.parquet.tmp')
exported_rows = 0
if status_counts['accepted'] == 0:
    raise ValueError('No accepted features to export.')
with pq.ParquetWriter(temporary_export, FEATURE_SCHEMA, compression='zstd') as writer:
    for part in sorted(OUTPUT_DIR.glob('part-*.parquet')):
        table = pq.read_table(part)
        accepted = table.filter(pc.equal(table['status'], 'accepted')).select(FEATURE_SCHEMA.names)
        accepted = accepted.cast(FEATURE_SCHEMA)
        if accepted.num_rows:
            writer.write_table(accepted)
            exported_rows += accepted.num_rows
if exported_rows != status_counts['accepted']:
    raise ValueError('Exported row count differs from checkpoint count.')
os.replace(temporary_export, parquet_path)
print(f'Saved {exported_rows:,} images to {parquet_path}')

## Step 6: Optional Quick-Loader
After extraction/export, switch to a high-RAM CPU runtime for modeling. Mount Drive again and use the exported file in the nested-CV notebook. This optional loader is disabled by default so Run All does not allocate full-dataset feature matrices during GPU extraction. Enable it only when you want to inspect the matrices.

In [ ]:
LOAD_FEATURE_MATRICES = False
if LOAD_FEATURE_MATRICES:
    loaded_ds = Dataset.from_parquet(str(parquet_path))
    X_clip = np.array(loaded_ds['clip_embedding'], dtype=np.float32)
    X_struct = np.array(loaded_ds['structural_features'], dtype=np.float32)
    X_combined = np.hstack([X_clip, X_struct])
    y_mean = np.array(loaded_ds['mean_score'], dtype=np.float32)
    y_std = np.array(loaded_ds['std_score'], dtype=np.float32)
    print(f'X_combined: {X_combined.shape}; y_mean: {y_mean.shape}')
else:
    print(f'Features are saved at {parquet_path}; matrix loading is disabled.')